In [0]:
%pip install --upgrade databricks-sdk requests
dbutils.library.restartPython()

In [0]:
import os
import json
import hashlib
import requests
from pathlib import Path
from databricks.sdk import WorkspaceClient

# Job parameters
dbutils.widgets.text("app_name", "mcp-hi-genie-orchestrator-dev")
dbutils.widgets.text("secret_scope_name", "hi_genie_credentials")
dbutils.widgets.text("client_id_dbs_key", "client_id_dev_matthew_giglia_hi_genie")
dbutils.widgets.text("client_secret_dbs_key", "client_secret_dev_matthew_giglia_hi_genie")

app_name = dbutils.widgets.get("app_name")
secret_scope = dbutils.widgets.get("secret_scope_name")
client_id_key = dbutils.widgets.get("client_id_dbs_key")
client_secret_key = dbutils.widgets.get("client_secret_dbs_key")

# Resolve app URL
w = WorkspaceClient()
app_info = w.apps.get(app_name)
app_url = app_info.url.rstrip("/")
print(f"App URL: {app_url}")

# Get M2M token for the App SPN
client_id = dbutils.secrets.get(secret_scope, client_id_key)
client_secret = dbutils.secrets.get(secret_scope, client_secret_key)
workspace_host = w.config.host.rstrip("/")

token_resp = requests.post(
    f"{workspace_host}/oidc/v1/token",
    data={"grant_type": "client_credentials", "scope": "all-apis"},
    auth=(client_id, client_secret),
)
token_resp.raise_for_status()
access_token = token_resp.json()["access_token"]
print("M2M token acquired successfully.")

In [0]:
# Read skill fixture files from the workspace
# The skill content lives in the hi-genie-orchestrator/skill/ folder
skill_root = Path("/Workspace/Users/matthew.giglia@databricks.com/brainstorming-hi-genie-orchestrator/genie-workstream-orchestrator/hi-genie-orchestrator/skill")

# Build manifest by walking the skill directory
sections = []

# Map of directory + filename -> section_path
skill_files = [
    # Entry point
    ("SKILL.md", "core/entry-point"),
    # Core
    ("core/state-machine.md", "core/state-machine"),
    # Roles
    ("roles/architect.md", "roles/architect"),
    ("roles/supervisor.md", "roles/supervisor"),
    ("roles/worker.md", "roles/worker"),
    ("roles/reviewer.md", "roles/reviewer"),
    # Capabilities
    ("capabilities/prompt-synthesis.md", "capabilities/prompt-synthesis"),
    ("capabilities/error-handling.md", "capabilities/error-handling"),
    ("capabilities/governance.md", "capabilities/governance"),
    ("capabilities/detection-fallback.md", "capabilities/detection-fallback"),
    # Backends
    ("backends/bundle.md", "backends/bundle"),
    ("backends/notebook.md", "backends/notebook"),
    ("backends/sql.md", "backends/sql"),
    ("backends/pipeline.md", "backends/pipeline"),
    ("backends/dashboard.md", "backends/dashboard"),
    ("backends/genie-space.md", "backends/genie-space"),
    ("backends/custom.md", "backends/custom"),
    # Surfaces
    ("surfaces/mcp-tools.md", "surfaces/mcp-tools"),
    ("surfaces/api-routes.md", "surfaces/api-routes"),
    ("surfaces/lakebase.md", "surfaces/lakebase"),
    ("surfaces/api.md", "surfaces/api"),
    ("surfaces/mcp.md", "surfaces/mcp"),
    # Examples
    ("examples/bootstrap-templates.md", "examples/bootstrap-templates"),
]

for file_path, section_path in skill_files:
    full_path = skill_root / file_path
    if full_path.exists():
        content = full_path.read_text(encoding="utf-8")
        content_hash = hashlib.sha256(content.encode()).hexdigest()
        sections.append({
            "section_path": section_path,
            "content": content,
        })
        print(f"  ✓ {section_path} ({len(content):,} chars, hash: {content_hash[:12]}...)")
    else:
        print(f"  ✗ {section_path} — file not found: {full_path}")

print(f"\nTotal sections: {len(sections)}")

In [0]:
# POST to /api/v1/skills/publish
headers = {
    "Authorization": f"Bearer {access_token}",
    "Content-Type": "application/json",
}

payload = {
    "sections": sections,
    "published_by": "deploy-job",
}

resp = requests.post(
    f"{app_url}/api/v1/skills/publish",
    headers=headers,
    json=payload,
    timeout=60,
)

if resp.status_code == 201:
    result = resp.json()
    print(f"\n✅ Skill published successfully!")
    print(f"   Version: {result['version']}")
    print(f"   Sections: {result['sections_published']}")
    print(f"   Published by: {result['published_by']}")
    print(f"   Published at: {result['published_at']}")
else:
    print(f"\n❌ Publish failed: HTTP {resp.status_code}")
    print(resp.text)
    raise Exception(f"Skill publish failed with status {resp.status_code}: {resp.text}")